# Import Libraries

In [1]:
import boto3
import requests
import json
from statistics import mean

# Config

In [2]:
bedrock = boto3.client("bedrock-runtime", region_name="us-west-2")
model_id = "global.anthropic.claude-sonnet-4-20250514-v1:0"

SUPABASE_URL = "https://zqvjzukfbfcixxaplzyp.supabase.co"
SUPABASE_KEY = "sb_publishable_LHNy1oE6UxRZtndOBjPoVg_bRaz6_VG"
headers = {
    "apikey":        SUPABASE_KEY,
    "Authorization": f"Bearer {SUPABASE_KEY}",
    "Content-Type":  "application/json"
}


# Helper Functions

In [3]:
def add_user_message(messages, content):
    if isinstance(content, str):
        messages.append({"role": "user", "content": [{"text": content}]})
    else:
        messages.append({"role": "user", "content": content})

def add_assistant_message(messages, content):
    if isinstance(content, str):
        messages.append({"role": "assistant", "content": [{"text": content}]})
    else:
        messages.append({"role": "assistant", "content": content})

def chat(messages, system=None, temperature=0.0, stop_sequences=[], tools=None):
    params = {
        "modelId":         model_id,
        "messages":        messages,
        "inferenceConfig": {
            "temperature":   temperature,
            "stopSequences": stop_sequences
        },
    }
    if system:
        params["system"] = [{"text": system}]
    if tools:
        params["toolConfig"] = {"tools": tools, "toolChoice": {"auto": {}}}

    response = bedrock.converse(**params)
    parts    = response["output"]["message"]["content"]
    return {
        "parts":       parts,
        "stop_reason": response["stopReason"],
        "text":        "\n".join([p["text"] for p in parts if "text" in p]),
    }


# Creating a retrival tool for the model

In [4]:
query_table_schema = {
    "toolSpec": {
        "name": "query_table",
        "description": """Queries the enriched_movies table in Supabase.
        Use this to fetch movies based on filters for recommendations,
        comparisons, or user preference analysis.
        Available columns:
        - movieId, title, overview, genres, releaseDate, runtime
        - budget, revenue, budget_tier (low/medium/high/blockbuster)
        - revenue_tier (flop/average/hit/blockbuster)
        - avg_rating, rating_count, user_ratings (JSON list of userId + rating)
        - sentiment (positive/negative/neutral)
        - audience_appeal (family/teen/adult/general)
        - production_effectiveness_score (0-100)
        """,
        "inputSchema": {
            "json": {
                "type": "object",
                "properties": {
                    "filters": {
                        "type": "object",
                        "description": """Key-value pairs to filter movies.
                        Examples:
                        {"sentiment": "positive", "budget_tier": "blockbuster"}
                        {"audience_appeal": "family", "revenue_tier": "hit"}
                        {"title": "Mr. Deeds"}
                        Pass empty {} to retrieve all movies.
                        """,
                        "additionalProperties": True
                    },
                    "limit": {
                        "type": "integer",
                        "description": "Max number of rows to return. Default 10.",
                        "default": 10
                    },
                    "order_by": {
                        "type": "string",
                        "description": "Column to sort results by. e.g. avg_rating, production_effectiveness_score, budget"
                    },
                    "ascending": {
                        "type": "boolean",
                        "description": "Sort ascending if true, descending if false. Default false.",
                        "default": False
                    }
                },
                "required": ["filters"]
            }
        }
    }
}





def query_table(filters, limit=10, order_by=None, ascending=False):
    url    = f"{SUPABASE_URL}/rest/v1/enriched_movies"
    params = {"limit": limit}

    # Build filter query params
    for col, val in filters.items():
        params[col] = f"eq.{val}"

    # Ordering
    if order_by:
        direction    = "asc" if ascending else "desc"
        params["order"] = f"{order_by}.{direction}"

    response = requests.get(url, headers=headers, params=params)

    if response.status_code == 200:
        return response.json()
    else:
        return {"error": f"{response.status_code} - {response.text}"}



        
def run_tool(tool_name, tool_input):
    if tool_name == "query_table":
        return query_table(
            filters   = tool_input.get("filters", {}),
            limit     = tool_input.get("limit", 10),
            order_by  = tool_input.get("order_by", None),
            ascending = tool_input.get("ascending", False)
        )
    else:
        raise Exception(f"Unknown tool: {tool_name}")


        

def run_tools(parts):
    tool_results = []
    for part in parts:
        if "toolUse" not in part:
            continue
        tool_use_id = part["toolUse"]["toolUseId"]
        tool_name   = part["toolUse"]["name"]
        tool_input  = part["toolUse"]["input"]
        try:
            output = run_tool(tool_name, tool_input)
            tool_results.append({
                "toolResult": {
                    "toolUseId": tool_use_id,
                    "content":   [{"text": json.dumps(output)}],
                    "status":    "success"
                }
            })
        except Exception as e:
            tool_results.append({
                "toolResult": {
                    "toolUseId": tool_use_id,
                    "content":   [{"text": f"Error: {e}"}],
                    "status":    "error"
                }
            })
    return tool_results


# Enabling the model to reason by collecting information in multiple steps

In [9]:
def run_conversation(messages, system=None):
    tools = [query_table_schema]
    while True:
        result = chat(messages, system=system, tools=tools)
        add_assistant_message(messages, result["parts"])
        if result["text"]:
            print(result["text"])
        if result["stop_reason"] != "tool_use":
            break
        tool_results = run_tools(result["parts"])
        add_user_message(messages, tool_results)
    return messages

# SYSTEM_PROMPT

In [10]:
SYSTEM_PROMPT = """You are an intelligent Movie Assistant with access to a curated database
of 100 enriched movies. You help users with three tasks:

## 1. Personalized Movie Recommendations
When a user asks for recommendations:
- Use query_table with relevant filters (sentiment, budget_tier, revenue_tier, audience_appeal, genres)
- Rank results by production_effectiveness_score or avg_rating
- Return top 5 movies with title, genres, avg_rating, and a reason for recommendation

## 2. User Preference Summary
When a user provides a userId:
- Fetch all movies and look through user_ratings column for that userId
- Identify their highly rated movies (rating >= 4.0) vs poorly rated (rating <= 2.0)
- Summarize their taste: preferred genres, sentiment, audience_appeal, budget_tier
- Give a 2-3 sentence profile of this user's movie preferences

## 3. Comparative Analysis
When a user wants to compare movies:
- Fetch each movie by title using query_table
- Compare across: budget, revenue, avg_rating, production_effectiveness_score, sentiment
- Declare a winner per category and an overall winner

## Guidelines
- Always query the database before answering — never guess movie details
- Be concise and structured in your responses
- If no movies match the filters, relax one filter and try again
- Always show avg_rating and production_effectiveness_score in your responses
"""

# Let's Test it with a simple query

In [11]:
test_query =  "Summarize preferences for userId 23"

messages = []
add_user_message(messages, test_query)
run_conversation(messages, system=SYSTEM_PROMPT)
print('------')

I'll analyze the movie preferences for userId 23 by examining their ratings across all movies in the database.
Now I'll analyze userId 23's ratings by looking through the user_ratings data for their specific ratings:

Looking through the data, I can see userId 23 has rated the following movies:

1. **Mr. Deeds** (2022) - Rating: 4.0 - Comedy/Romance, positive sentiment, high budget, blockbuster revenue
2. **The Elephant Man** (1955) - Rating: 4.0 - Drama/History, neutral sentiment, low budget, average revenue  
3. **Traffic** (1900) - Rating: 3.5 - Thriller/Drama/Crime, negative sentiment, medium budget, blockbuster revenue
4. **Lethal Weapon 2** (942) - Rating: 4.0 - Action/Adventure/Comedy/Crime/Thriller, positive sentiment, medium budget, blockbuster revenue
5. **Sabrina** (6620) - Rating: 3.5 - Comedy/Drama/Romance, positive sentiment, low budget, average revenue
6. **Manderlay** (1951) - Rating: 3.0 - Drama, negative sentiment, medium budget, flop revenue
7. **Heat** (949) - Ratin

# The Result is good but Let's eveluate our Model using a Model Based Grading Technique

In [12]:
def fetch_data_snapshot():
    movies   = query_table(filters={}, limit=100, order_by="production_effectiveness_score")  # ← all 100
    snapshot = []
    for m in movies:
        snapshot.append({
            "title":           m.get("title"),
            "genres":          m.get("genres"),
            "sentiment":       m.get("sentiment"),
            "budget_tier":     m.get("budget_tier"),
            "revenue_tier":    m.get("revenue_tier"),
            "audience_appeal": m.get("audience_appeal"),
            "avg_rating":      m.get("avg_rating"),
            "production_effectiveness_score": m.get("production_effectiveness_score"),
            "sample_userIds":  [r["userId"] for r in json.loads(m.get("user_ratings") or "[]")][:3]
        })
    return snapshot

# Generating Evaluation Dataset

In [13]:
def generate_dataset(snapshot):
    snapshot_text = json.dumps(snapshot, indent=2)

    prompt = f"""
    You are designing an evaluation dataset for a Movie AI Assistant.
    The assistant handles 3 types of queries:
    1. Recommendations - e.g. "Recommend action movies with positive sentiment"
    2. User Preference  - e.g. "Summarize preferences for userId 23"
    3. Comparative      - e.g. "Compare Movie A vs Movie B on budget and effectiveness"

    Here is the COMPLETE list of all movies in the database:
    <data_snapshot>
    {snapshot_text}
    </data_snapshot>

    Using ONLY real titles, real userIds, and real filter values from the snapshot above,
    generate 10 test cases (3 recommendations, 3 user_preference, 4 comparisons).

    Output format:
    ```json
    [
        {{
            "query":             "The natural language query to send to the assistant",
            "query_type":        "recommendation" or "user_preference" or "comparison",
            "solution_criteria": "What a good response must include to be correct"
        }}
    ]
    ```

    Rules:
    - Use ONLY movie titles that exist in the snapshot above — do not invent titles
    - Use ONLY userIds that appear in sample_userIds in the snapshot
    - Use ONLY filter values (sentiment, budget_tier etc.) that exist in the snapshot
    - The evaluator will have access to this same snapshot, so all titles must be verifiable
    - Make queries natural and varied
    """

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    result = chat(messages, stop_sequences=["```"], temperature=1.0)  
    return json.loads(result["text"])

# Creatig a Model Based Grader

In [14]:
def run_prompt(test_case):
    return run_conversation(test_case["query"])


def grade_by_model(test_case, output, snapshot):
    eval_prompt = f"""
    You are an expert evaluator for a Movie AI Assistant.
    The assistant has access to a database of 100 movies. Here is the complete list:

    <data_snapshot>
    {json.dumps(snapshot, indent=2)}
    </data_snapshot>

    Query sent to the assistant:
    <query>
    {test_case["query"]}
    </query>

    Query type: {test_case["query_type"]}

    Assistant response:
    <response>
    {output}
    </response>

    Evaluation criteria:
    <criteria>
    {test_case["solution_criteria"]}
    </criteria>

    Important: Only flag something as hallucination if the title or userId does NOT appear 
    in the data_snapshot above. The assistant has access to all 100 movies.

    Evaluate the response and return a JSON object:
    {{
        "strengths":  ["strength 1", "strength 2"],
        "weaknesses": ["weakness 1", "weakness 2"],
        "reasoning":  "concise explanation",
        "score":      number between 1-10
    }}
    
    Scoring guide:
    - 9-10: Fully addresses query, accurate data, well structured
    - 7-8 : Mostly correct, minor gaps
    - 5-6 : Partially correct, missing key elements
    - 1-4 : Incorrect, hallucinated data, or missed the query type
    """

    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")
    result = chat(messages, stop_sequences=["```"])
    return json.loads(result["text"])


# Eveluation functions

In [15]:
def run_test_case(test_case, snapshot):
    print(f"\n  Running: {test_case['query']}.")
    output = run_prompt(test_case)
    grade  = grade_by_model(test_case, output, snapshot)  

    return {
        "query":      test_case["query"],
        "query_type": test_case["query_type"],
        "output":     output,
        "score":      grade["score"],
        "strengths":  grade["strengths"],
        "weaknesses": grade["weaknesses"],
        "reasoning":  grade["reasoning"],
    }


def run_eval(dataset, snapshot):
    results = []
    for test_case in dataset:
        result = run_test_case(test_case, snapshot)  
        results.append(result)
        print(f"  Score: {result['score']}/10 — {result['reasoning']}.")

    avg = mean([r["score"] for r in results])

    print("\n" + "=" * 60)
    print(f"  EVALUATION COMPLETE")
    print(f"  Average Score : {avg:.1f} / 10")
    print(f"  Total Cases   : {len(results)}")
    print("=" * 60)

    for qtype in ["recommendation", "user_preference", "comparison"]:
        subset = [r for r in results if r["query_type"] == qtype]
        if subset:
            type_avg = mean([r["score"] for r in subset])
            print(f"  {qtype:<20} avg: {type_avg:.1f}/10  ({len(subset)} cases)")

    return results, avg

# Updating run_conversation Function for testing purposes 

In [17]:
def run_conversation(query):
    messages = []
    add_user_message(messages, query)
    tools = [query_table_schema]

    while True:
        result = chat(messages, system=SYSTEM_PROMPT, tools=tools)
        add_assistant_message(messages, result["parts"])

        if result["stop_reason"] != "tool_use":
            break

        tool_results = run_tools(result["parts"])
        add_user_message(messages, tool_results)

    return result["text"]

# Final Evaluation Test

In [18]:
print("1: Fetching data snapshot from Supabase...")
snapshot = fetch_data_snapshot()
print(f"  Fetched {len(snapshot)} movies")

print("\n2: Generating evaluation dataset...")
dataset = generate_dataset(snapshot)
print(f"  Generated {len(dataset)} test cases")
with open("eval_dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)

print("\n3: Running evaluation...")
results, avg_score = run_eval(dataset, snapshot)  # ← pass snapshot
with open("eval_results.json", "w") as f:
    json.dump({"average_score": avg_score, "results": results}, f, indent=2)

print("\nResults saved to eval_results.json")

1: Fetching data snapshot from Supabase...
  Fetched 100 movies

2: Generating evaluation dataset...
  Generated 10 test cases

3: Running evaluation...

  Running: Recommend action movies with positive sentiment.
  Score: 7/10 — The assistant successfully identified and recommended the three key action movies with positive sentiment mentioned in the criteria, providing accurate data and good explanations. However, 2 out of 5 recommendations (The Mask and Hook) don't actually have 'Action' listed in their genres in the database, which is a significant accuracy issue for a recommendation query..

  Running: Find me some horror movies that are suitable for families.
  Score: 7/10 — The assistant correctly identifies Poltergeist as the only family-friendly horror movie in the database and provides accurate data. However, it dilutes the response by recommending non-horror movies as alternatives, which doesn't fully address the specific query for 'horror movies.' While the creative alternat

# Based on the results we can Understand that the model is struggling with some hallucinations and  Budget Calculation.
## Therefore let's update the system prompt by adding some grounding rules, constrains and guidelines.
## Let's also improve User Preference Summary by adding a Chain of thought prompting technique specifically for user_preference related questions.

In [28]:
SYSTEM_PROMPT = """You are an intelligent Movie Assistant with access to a curated database
of 100 enriched movies. You help users with three tasks:

## 1. Personalized Movie Recommendations
- Use query_table with relevant filters (sentiment, budget_tier, revenue_tier, audience_appeal)
- Rank by production_effectiveness_score or avg_rating
- Return top 5 movies with title, genres, avg_rating, and a reason for recommendation
- STRICT GENRE RULE: If the user asks for a specific genre (e.g. "action"),
  only recommend movies whose genres field explicitly contains that genre word.
  Do NOT include movies that are adjacent (e.g. Thriller is not Action).
- If fewer than 5 movies match all filters, reduce the number of recommendations
  rather than including movies that don't match.

## 2. User Preference Summary
When given a userId, follow these steps EXACTLY:

Step 1 — Fetch ALL movies from the database (use filters={}, limit=100)
Step 2 — For each movie, the user_ratings column is a JSON string like:
         '[{{"userId": 23, "rating": 4.0}}, {{"userId": 45, "rating": 3.0}}]'
         Search through each movie's user_ratings to find entries where userId matches.
Step 3 — Collect ALL movies this user has rated. For each, note:
         - title, genres, sentiment, budget_tier, audience_appeal
         - the specific rating this user gave (not avg_rating)
Step 4 — Split into:
         - Liked movies: rating >= 4.0
         - Disliked movies: rating <= 2.0
         - Neutral movies: rating between 2.0 and 4.0
Step 5 — Summarize preferences:
         - Favorite genres (most common in liked movies)
         - Preferred sentiment, budget_tier, audience_appeal
         - Average rating this user gives
         - 2-3 sentence personality profile of this user's taste
- If the userId has rated fewer than 3 movies, say so clearly and summarize what little is available.
- NEVER confuse avg_rating (the global average) with the user's personal rating.

## 3. Comparative Analysis
- Fetch each movie by title using query_table
- Compare ONLY using fields that exist in the database:
  budget_tier, revenue_tier, avg_rating, rating_count, production_effectiveness_score,
  sentiment, audience_appeal, genres
- NEVER invent or estimate specific dollar amounts for budget or revenue.
  The database only stores tiers (low/medium/high/blockbuster) — use those exact labels.
- NEVER calculate ROI, return multiples, or revenue/budget ratios — this data is not available.
- NEVER add release years or runtime minutes unless explicitly returned by query_table.
- Declare a winner per category and an overall winner based only on available data.

## Guidelines
- Always query the database before answering — never guess or invent movie details
- NEVER fabricate specific numbers not returned by query_table
- If a genre filter returns no results, tell the user clearly and suggest the closest alternative
- Always show avg_rating and production_effectiveness_score in your responses
- Stick strictly to what the database returns — do not supplement with outside knowledge
"""

# Re-eveluating the Model on the same dataset

In [29]:
print("\n3: Running evaluation...")
results, avg_score = run_eval(dataset, snapshot)  # ← pass snapshot
with open("eval_results.json", "w") as f:
    json.dump({"average_score": avg_score, "results": results}, f, indent=2)

print("\nResults saved to eval_results.json")


3: Running evaluation...

  Running: Recommend action movies with positive sentiment.
  Score: 8/10 — The assistant successfully identified and recommended 5 out of 7 action movies with positive sentiment from the database. All data provided was accurate and matched the database exactly. The response was well-formatted and included thoughtful explanations for each recommendation. While not completely comprehensive, it covered the most relevant examples and specifically included all movies mentioned in the evaluation criteria..

  Running: Find me some horror movies that are suitable for families.
  Score: 4/10 — The assistant failed to identify 'Poltergeist' as a horror movie with family audience appeal, which directly answers the user's query. While the alternative recommendations are helpful, the primary request was not fulfilled when there was actually a valid option available in the database..

  Running: Show me blockbuster movies with low budget that performed really well.
  Sco

# The Quality of the Results are Significantly Better 

In [30]:
query = "Recommend action movies with high revenue and positive sentiment"
print(run_conversation(query))

Perfect! Now I can see the action movies with positive sentiment and high revenue. Let me identify the ones that specifically contain "Action" in their genres and have high revenue (blockbuster tier):

Based on my search of the database, I found **3 action movies** that match your criteria of having positive sentiment and high revenue (blockbuster tier). Here are my top recommendations:

## **Top 3 Action Movie Recommendations**

### 1. **Lethal Weapon 2** (1989)
- **Genres:** Action, Adventure, Comedy, Crime, Thriller
- **Average Rating:** 3.88/5 ⭐
- **Production Effectiveness Score:** 90/100
- **Revenue Tier:** Blockbuster
- **Budget Tier:** Medium
- **Why I recommend it:** This action-packed sequel delivers explosive entertainment with Martin Riggs and Roger Murtaugh's chemistry. The positive sentiment and strong audience appeal make it a crowd-pleaser that successfully balances intense action with comedic moments.

### 2. **Indiana Jones and the Last Crusade** (1989)
- **Genres:** 